In [ ]:
from model_zoo import (
    ModelZoo, ModelRegistry, MetricRegistry,
    ModelSpec, ModelKind, EvalMetricSpec, MetricKind, MetricDirection,
    HyperParamSpace, HyperParamField, DType,
    ExecutionBinding, BindingKind,
    DataRef, DataRefType, ExecutionContext,
    ModelRunRequest, MetricComputeRequest, EvaluationSpec,
)
from model_zoo.runners.library import LibraryModelRunner, LibraryMetricRunner
from model_zoo.examples.models import make_synthetic_linear
from model_zoo import r_style_summary_string
from model_zoo.utils.logging import configure_logging, get_logger
from model_zoo.examples.sklearn_models import sk_lr_train_and_infer  # noqa: F401 (for binding importability)
from model_zoo.examples.pymc_models import pymc_linear_nuts  # noqa: F401 (for binding importability)
from model_zoo.examples import metrics as example_metrics
from model_zoo.contracts.validation import ParamValidationService
from model_zoo import DefaultParamValidator
from model_zoo.contracts.audit import AuditLog, AuditEvent

class PrintAudit(AuditLog):
    def append(self, event: AuditEvent) -> None:
        print(f"[AUDIT] {event.kind} ref={event.ref_id}")

In [3]:
class NoOpValidator(ParamValidationService):
    def validate_model_params(self, model_spec, params): return None
    def validate_metric_params(self, metric_spec, params): return None

class PrintAudit(AuditLog):
    def append(self, event: AuditEvent) -> None:
        print(f"[AUDIT] {event.kind} ref={event.ref_id}")

In [4]:
model_reg, metric_reg = ModelRegistry(), MetricRegistry()
zoo = ModelZoo(model_reg, metric_reg,
                   binding_registry=None,  # not needed; we pass binding on request in this demo
                   model_runner=LibraryModelRunner(),
                   metric_runner=LibraryMetricRunner(),
                   validator=NoOpValidator(),
                   audit=PrintAudit())

In [6]:
lin_hp = HyperParamSpace(fields={
        "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True, required=False,
                                          description="Include intercept in linear model"),
    })

bayes_hp = HyperParamSpace(fields={
        "sigma": HyperParamField("sigma", DType.FLOAT, default=0.5, required=True,
                                  description="Observation noise std"),
        "prior_tau": HyperParamField("prior_tau", DType.FLOAT, default=10.0, required=False,
                                      description="Prior std for slope"),
        "fit_intercept": HyperParamField("fit_intercept", DType.BOOL, default=True, required=False),
    })

In [7]:
lin_spec = ModelSpec(
        model_id="linear_regression_v1",
        name="LinearRegression_OLS",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=lin_hp,
        tags=["linear", "ols"],
        metadata={"example": True}
    )

bayes_spec = ModelSpec(
        model_id="bayes_linear_v1",
        name="BayesianLinearRegression_Conjugate",
        version="1.0.0",
        kind=ModelKind.LIBRARY,
        hyperparams=bayes_hp,
        tags=["bayesian", "regression"],
        metadata={"example": True}
    )

In [8]:
zoo.register_model(lin_spec, actor="you")
zoo.register_model(bayes_spec, actor="you")

[AUDIT] REGISTER_MODEL ref=linear_regression_v1
[AUDIT] REGISTER_MODEL ref=bayes_linear_v1


In [9]:
r2_spec = EvalMetricSpec(
        metric_id="r2_v1",
        name="R2_Score",
        version="1.0.0",
        direction=MetricDirection.HIGHER_IS_BETTER,
        kind=MetricKind.LIBRARY,
        tags=["regression", "goodness_of_fit"],
    )
mse_spec = EvalMetricSpec(
        metric_id="mse_v1",
        name="MeanSquaredError",
        version="1.0.0",
        direction=MetricDirection.LOWER_IS_BETTER,
        kind=MetricKind.LIBRARY,
        tags=["regression", "error"],
    )
zoo.register_metric(r2_spec, actor="you")
zoo.register_metric(mse_spec, actor="you")

[AUDIT] REGISTER_METRIC ref=r2_v1
[AUDIT] REGISTER_METRIC ref=mse_v1


In [10]:
# List models
print("Registered models:")
for m in zoo.list_models():
        print(" -", m.model_id, m.name, m.version)
print("Registered metrics:")
for mt in zoo.list_metrics():
        print(" -", mt.metric_id, mt.name, mt.version)

Registered models:
 - linear_regression_v1 LinearRegression_OLS 1.0.0
 - bayes_linear_v1 BayesianLinearRegression_Conjugate 1.0.0
Registered metrics:
 - r2_v1 R2_Score 1.0.0
 - mse_v1 MeanSquaredError 1.0.0


In [11]:
# Prepare synthetic dataset (INLINE DataRefs)
X, y = make_synthetic_linear(n=100, slope=2.0, intercept=1.0, noise_sigma=0.5)
X_ref = DataRef(ref_type=DataRefType.INLINE, inline=X)
y_ref = DataRef(ref_type=DataRefType.INLINE, inline=y)

# Train/test split indices (80/20)
n = len(X)
split = int(0.8 * n)
train_idx = list(range(0, split))
test_idx = list(range(split, n))


In [13]:
lin_binding = ExecutionBinding(
        bound_id="linear_regression_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.models", callable="linreg_train_and_infer",
    )
bayes_binding = ExecutionBinding(
        bound_id="bayes_linear_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.models", callable="bayes_linreg_train_and_infer",
    )

In [14]:
r2_binding = ExecutionBinding(
        bound_id="r2_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.metrics", callable="r2_score",
    )
mse_binding = ExecutionBinding(
        bound_id="mse_v1", kind=BindingKind.LIBRARY,
        module="model_zoo.examples.metrics", callable="mse",
    )

In [15]:
# Run linear regression with training on train_idx and evaluate on test_idx
lin_req = ModelRunRequest(
        model_id="linear_regression_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={"fit_intercept": True, "train_indices": train_idx},
        context=ExecutionContext(random_seed=2025),
        dataset_version="synthetic:v1",
        binding=lin_binding,
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            test_indices=test_idx,
            bindings={
                "r2_v1": r2_binding,
                "mse_v1": mse_binding,
            }
        ),
    )

In [16]:
def wait_for_result(zoo_obj, run_id: str, actor: str = "you", max_tries: int = 100):
        import time
        res = None
        for _ in range(max_tries):
            res = zoo_obj.get_run_result(run_id, actor=actor)
            if not res.metadata.get("in_progress"):
                return res
            time.sleep(0.01)
        return res

In [17]:
lin_handle = zoo.submit_run(lin_req, actor="you")

[AUDIT] RUN_SUBMIT ref=linear_regression_v1


In [19]:
lin_res = wait_for_result(zoo, lin_handle.run_id, actor="you")
print("LinearRegression: status=", lin_res.status.value, "coefficients=", lin_res.diagnostics)

# Metrics computed automatically based on evaluation spec
print("LinearRegression metrics:", lin_res.eval_scores)

[AUDIT] RUN_RESULT ref=run_20251009_083806_014028
LinearRegression: status= succeeded coefficients= {'slope': 1.9828832677020547, 'intercept': 1.1432151220307478}
LinearRegression metrics: {'r2_v1': 0.8872138886194473, 'mse_v1': 0.20072719667134145}


In [20]:
bayes_req = ModelRunRequest(
        model_id="bayes_linear_v1",
        inputs={"X": X_ref, "y": y_ref},
        params={"sigma": 0.5, "prior_tau": 5.0, "fit_intercept": True, "train_indices": train_idx},
        context=ExecutionContext(random_seed=2025),
        dataset_version="synthetic:v1",
        binding=bayes_binding,
        evaluation=EvaluationSpec(
            metric_ids=["r2_v1", "mse_v1"],
            y_true_key="y",
            test_indices=test_idx,
            bindings={
                "r2_v1": r2_binding,
                "mse_v1": mse_binding,
            }
        ),
    )

In [21]:
bayes_handle = zoo.submit_run(bayes_req, actor="you")
bayes_res = wait_for_result(zoo, bayes_handle.run_id, actor="you")
print("BayesLinear: status=", bayes_res.status.value, "params=", bayes_res.diagnostics)
print("BayesLinear metrics:", bayes_res.eval_scores)

[AUDIT] RUN_SUBMIT ref=bayes_linear_v1
[AUDIT] RUN_RESULT ref=run_20251009_084334_086926
[AUDIT] RUN_RESULT ref=run_20251009_084334_086926
BayesLinear: status= succeeded params= {'posterior_mean_beta': 1.9828367877027144, 'posterior_var_beta': 0.000586015330163381, 'alpha': 1.1434033660280765}
BayesLinear metrics: {'r2_v1': 0.8872257168809626, 'mse_v1': 0.20070614573034873}
